# Tickable 6 - model solution

*&#169; Eike Mueller, University of Bath 2019-2026. This model solution is copyright of Eike Mueller, University of Bath. It is provided exclusively for educational purposes at the University and is to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## &#9745; Task 1
To implement the fast power method for matrices, we can take the algorithm from the lectures (which raises a number $x$ to some power $n$) and simply replace all ordinary multiplications by matrix multiplications. We also need to take care to return the identity matrix if $n=0$. This results in the following pseudocode:

### Function $\operatorname{fast\_power\_mat}(A,n)$
**Input**: $m\times m$ matrix $A$, Integer power $n$;
**Output** $A^{n}$
1. **if** $n==0$ **then**
1. $~~~~$ **return** $I_{m\times m}$
1. **else if** $n==1$ **then**
1. $~~~~$ **return** $A$
1. **else if** $n$ is even **then**
1. $~~~~$ **return** $\operatorname{fast\_power\_mat}(A\circ A,n/2)$
1. **else**
1. $~~~~$ **return** $A\circ \operatorname{fast\_power\_mat}(A \circ A,(n-1)/2)$
1. **end if**

The corresponding Python code is this:

In [ ]:
import numpy as np

def fast_power_mat(A,n):
    '''Compute matrix power A^n using recursion
    
    input:
      * A matrix to raise to the n-th power
      * n power n
    output:
      A^n
    '''
    nrow, ncol = A.shape
    if not (n == int(n)):
        raise Exception('Can only raise matrix to integer powers.')
    elif (n < 0):
        raise Exception('Can only raise matrix to non-negative powers.')
    elif not (nrow == ncol):        
        raise Exception('Matrix must be square.')
    elif n==0:
        return np.identity(nrow)
    elif n==1:
        return A
    elif n == 2*(n//2):
        return fast_power_mat(A@A, n//2)
    else:
        return A @ fast_power_mat(A@A, (n-1)//2)

## &#9745; Task 2
Based on this, we can now compute the $n$-th Fibonacci number by first raising the Fibonacci matrix $A_\text{fib}=\begin{pmatrix}1&1\\1&0\end{pmatrix}$ to the $n$-th power with our `fast_power_mat()` function, and then multiplying it by $\boldsymbol{f}_0$ to obtain $\boldsymbol{f}_n=A_{\text{fib}}^n\boldsymbol{f}_0$. Finally, we extract the first component of the resulting vector $\boldsymbol{f}_n=\begin{pmatrix}F_n\\F_{n-1}\end{pmatrix}$.

This is implemented in the following Python code:

In [ ]:
def fib_rec_fast_mat(n):
    '''Compute the n-th Fibonacci number by extracting the first component
    of the n-th Fibonacci vector (F_n, F_{n-1}).

    The Fibonacci vector is computed by raising the matrix A_fib = [[1,1],[1,0]]
    to the n-th power with the fast_power_mat() method.

    input: Number n
    output: n-th Fibonacci number F_n
    '''
    if not (n == int(n)):
        raise Exception('Fibonacci number F_n not defined for non-integer values')
    elif (n < 0):
        raise Exception('Fibonacci number F_n not defined for negative values')
    A_fib = np.array([[1,1],[1,0]])
    f_fib0 = np.array([0,1])
    f_fib = fast_power_mat(A_fib,n) @ f_fib0
    return f_fib[0]

Again, all error checking is done by raising exceptions. Specifically, for `fast_power()` we check that:

1. the power $n$ is an integer
2. the power $n$ in non-negative
3. the matrix is a square matrix

## Further thoughts on efficiency
The `slow_power_mat()` function can be implemented as follows:

In [ ]:
def slow_power_mat(A,n):
    '''Compute matrix power A^n using recursion
    
    input:
      * A matrix to raise to the n-th power
      * n power n
    output:
      A^n
    '''
    nrow, ncol = A.shape
    if not (n == int(n)):
        raise Exception('Can only raise matrix to integer powers.')
    elif (n < 0):
        raise Exception('Can only raise matrix to non-negative powers.')
    elif not (nrow == ncol):        
        raise Exception('Matrix must be square.')
    elif n==0:
        return np.identity(nrow)
    elif n==1:
        return A
    else:
        return A @ slow_power_mat(A, n-1)

The following code measures the runtime of `fast_power_mat()` and `slow_power_mat()` for different values of $n$, while multiplying matrices of size $128\times 128$:

In [ ]:
import numpy as np
A = np.random.randint(low=1,high=128,size=(128,128))
print ('*** fast_power_mat() ***')
for n in (2,4,8,16,32,64):
    print ('n=',n)
    %timeit fast_power_mat(A,n)
print ('*** slow_power_mat() ***')
for n in (2,4,8,16,32,64):
    print ('n=',n)
    %timeit slow_power_mat(A,n)

Running the code on my computer I measure the following times (all numbers in the table are given in milliseconds):

| $n$ | time for `fast_power_mat()` | time for `slow_power_mat()`|
| --- | --- | --- |
| $2$ | 2.1 | 2.1 |
| $4$ | 4.2 | 6.3 |
| $8$ | 6.3 | 14.9 |
| $16$ | 8.3 | 32.2 |
| $32$ | 10.4 | 66.4 |
| $64$ | 12.5 | 135.0 |

If $n$ is doubled, the time it takes to execute `fast_power_mat()` increases by a constant (approximately $2\text{ms}$). This is in stark contrast to `slow_power_mat()`: here the times increases by a factor of two whenever $n$ is doubled. For $n=64$ `fast_power_mat()` is more than ten times as fast as `slow_power_mat()`.

For a given value of $n$, `slow_power_mat()` calls itself recursively $n-1$ times and each time we have to multiply two $m\times m$ matrices, which requires $(2m-1)m^2$ elops. Hence, the total number of elops is

$$
E_{\operatorname{slow\_power}}(n,m) = (n-1)(2m-1)m^2 \approx 2nm^3 \qquad\text{for $n,m\gg 1$.}
$$

In contrast, if $n=2^k$ (i.e. $n$ is a power of two) `fast_power_mat()` only calls itself recursively $k=\log_2 (n)$ times. Again, in each call we have to multiply two $m\times m$ matrices, so the total number of elops is

$$
E_{\operatorname{fast\_power}}(n,m) = \log_2(n) (2m-1)m^2 \approx \log_2(n) m^3 \qquad\text{for $m\gg 1$.}
$$

This explains the much slower growth in runtime for the `fast_power_mat()` function.